# L5: Image Generation Agent

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [ ]:
import json
import random
from google import genai
from google.genai import types
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from typing import List, Dict
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown, display

In [ ]:
import os
from helper import authenticate

credentials, project_id = authenticate()
client = genai.Client(
    project=project_id,
    location="global",
    credentials=credentials,
    http_options=types.HttpOptions(
         base_url=os.getenv("GOOGLE_VERTEX_BASE_URL")
    )
)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different run results:</b> The output generated by AI models, and agents can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different run results:</b> The output generated by AI models, and agents can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

In [ ]:
TEXT_MODEL_ID = "gemini-3-flash-preview"
IMAGE_MODEL_ID = "gemini-3.1-flash-image-preview"

## Tool 1: `brand_analysis`

In [ ]:
def brand_analysis(brand_guidelines_image: str):

    class BrandInfo(BaseModel):
        colors: str
        typography: str
        icon_style: str
        brand_voice: str
        ui_elements: str

    image_analysis_prompt = """
    Analyze the attached brand guidelines image and extract a comprehensive 
    Visual Design DNA for a high-fidelity UI mockup.

    Instructions:

    Color Palette: Identify primary, secondary, and accent HEX codes 
    (or detailed descriptions). Note the "vibe" of the colors.

    Typography: Identify font families, weights, and the intended hierarchy.

    Logo & Iconography: Describe the logo's geometry and the required style 
    for icons (e.g., "Thin-line minimalist," "3D Claymorphism," or "Solid 
    glyphs").

    Brand Voice: Define the "personality" of the brand in 5 keywords 
    (e.g., Trustworthy, Disruptive, Playful).

    UI Specifics: Extract any mentions of border-radii (rounded vs. sharp), 
    shadow usage, or spacing patterns.
    """

    with open(brand_guidelines_image, "rb") as f:
        image = f.read()

    response = client.models.generate_content(
        model=TEXT_MODEL_ID,
        contents=[
            types.Part.from_bytes(data=image, mime_type="image/png"),
            image_analysis_prompt,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=BrandInfo,
        ),
    )
    return json.loads(response.text)

### Tool 1 - Test

In [ ]:
brand_image = "guide.png"
display(Image(brand_image, width=400))

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> The following notebook cell may take a few minutes to run.</p>

In [ ]:
analysis = brand_analysis(brand_image)

sections = {
    "Color Palette": "colors",
    "Typography": "typography",
    "Logo & Iconography": "icon_style",
    "Brand Voice": "brand_voice",
    "UI Specifics": "ui_elements"
}

for title, key in sections.items():
    print(f"\n--- {title}: ---")
    display(Markdown(analysis.get(key, "No data found.")))

## Tool 2: `generate_design_concepts`

In [ ]:
def generate_design_concepts(
        user_description: str, 
        brand_guidelines_image: str, 
        brand_guidelines: str
) -> dict:

    class ConceptInfo(BaseModel):
        title: str
        description: str
        prompt: str

    class Ideas(BaseModel):
        ideas: List[ConceptInfo]

    prompt = f"""
    Based on the following user description and brand guidelines, generate a 
    single comprehensive UI design concept for a web application.

    User Description: {user_description}
    Brand Guidelines (extracted keywords): {brand_guidelines}

    For the concept, provide:
    * Title: A concise and evocative name for the design concept.
    * Design Description: A detailed description of the layout, color 
    palette, typography, and key visual elements, explaining how it aligns 
    with the user's description and brand identity.
    * Nano Banana Prompt: A specific text prompt that can be used with 
    Nano Banana to create a photorealistic or illustrative mockup UI 
    rendering of this concept. Focus on visual details.

    Ensure the concept fulfills all of the user's
    core requirements and maintains brand consistency.
    """

    with open(brand_guidelines_image, "rb") as f:
        image = f.read()

    response = client.models.generate_content(
        model=TEXT_MODEL_ID,
        contents=[
            types.Part.from_bytes(data=image, mime_type="image/png"),
            prompt,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=Ideas,
        ),
    )

    return json.loads(response.text)

### Tool 2 - Test

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> The following notebook cell may take a few minutes to run.</p>

In [ ]:
brief = """
Create a web UI landing page for this cooking app.
"""

ideas = generate_design_concepts(brief, "guide.png", analysis)

for concept in ideas['ideas']:
    print(f"\n--- Concept ---")
    display(Markdown(f"**Title:** {concept['title']}"))
    display(Markdown(f"**Description:** {concept['description']}"))
    display(Markdown(f"**Prompt:** {concept['prompt']}"))

## Tool 3: `generate_idea_image`

In [ ]:
def generate_idea_image(prompt: str, brand_guidelines_image: str) -> str:

    prompt = f"""
    Use the following description to generate a UI mockup that adheres 
    to the provided brand guidelines: {prompt}
    """
    with open(brand_guidelines_image, "rb") as f:
        image = f.read()

    response = client.models.generate_content(
        model=IMAGE_MODEL_ID,
        contents=[
            types.Part.from_bytes(
                data=image,
                mime_type="image/png",
            ),
            prompt,
        ],
        config=types.GenerateContentConfig(
            response_modalities=['IMAGE'],
            image_config=types.ImageConfig(
                aspect_ratio="16:9",
                output_mime_type="image/png",
            ),
        ),
    )
    for part in response.candidates[0].content.parts:
        if part.inline_data:
            id = random.randint(1, 10000)
            path = f"{id}_idea.png"
            with open(path, "wb") as f:
                f.write(part.inline_data.data)
            display(Image(data=part.inline_data.data, width=500))
            return path

### Tool 3 - Test

In [ ]:
prompt = concept['prompt']
print(prompt)

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> The following notebook cell may take a few minutes to run.</p>

In [ ]:
image_idea = generate_idea_image(prompt, "guide.png")
image_path = image_idea

## Tool 4: `evaluate_image`

In [ ]:
def evaluate_image(threshold: float, brand_guide: str, ui_image: str) -> dict:
        
    class EvaluationResult(BaseModel):
        scores: Dict[str, float]
        overall: float
        is_passing: bool = Field(alias="pass")
        feedback: str

    eval_prompt = f"""
    Carefully and critically evaluate the attached UI mockup against the 
    original brand guidelines.

    Score each criterion in a range from 1 (poor) to 5 (excellent):
    - visual aesthetic: overall appeal and brand alignment (compare logos, 
    colors, typography and brand elements against the guideline image)
    - contrast: unique elements should stand apart in regards to color, 
    shape, and direction
    - repetition: consistency of components and patterns
    - alignment: grid consistency and horizontal/vertical flow
    - proximity: logical grouping of related elements

    Return ONLY a JSON object:
    - scores: {{criterion_name: score, ...}},
    - overall: average of all scores,
    - is_passing: true if overall >= {threshold},
    - feedback: specific prompt changes to improve if fail, else 'Looks good'

    """
    with open(ui_image, "rb") as f:
        image = f.read()
    with open(brand_guide, "rb") as f:
        guide = f.read()

    response = client.models.generate_content(
        model=TEXT_MODEL_ID,
        contents=[
            types.Part.from_bytes(data=image, mime_type="image/png"),
            types.Part.from_bytes(data=guide, mime_type="image/png"),
            eval_prompt,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=EvaluationResult,
        ),
    )

    result = json.loads(response.text)
    print(f"Overall: {result['overall']}")
    print(f"Feedback: {result['feedback']}")
    return result

### Tool 4 - Test

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> The following notebook cell may take a few minutes to run.</p>

In [ ]:
results = evaluate_image(4.6, "guide.png", image_path)

print(f"--- Evaluation Result [{'PASS' if results['pass'] else 'FAIL'}] ---")
print(f"Overall Score: {results['overall']}")
for criteria, score in results['scores'].items():
    print(f"- {criteria.title()}: {score}")
print(f"\nFeedback: {results['feedback']}")

## Wiring the Agent

In [ ]:
SYSTEM_PROMPT = """You are a UI design agent. When asked to create a 
mockup UI:

1. Call brand_analysis with the user's style guide image to get an essence 
of the brand. The initial style image is in a local file named guide.png.
2. Call generate_design_concepts to get two distinct mockup UI concepts 
based on the user input and brand guidelines. If the resulting two concepts 
are not distinct generate a new concept until there are two different design 
ideas. Distinct concepts should refer to different themes, UI components, 
layouts, and elements.
3. For each concept:
   a. Call generate_idea_image with the prompt and style guide image filename "guide.png". 
   Make sure you remember the returned image file name for the evaluation step.
   b. Call evaluate_image with a 4.8 threshold. For brand_guide, ALWAYS pass 
   the string "guide.png" (the filename, not the brand analysis text). 
   For ui_image, pass the image filename returned by generate_idea_image.
   c. If the evaluation fails (pass=false), use the feedback to adjust the 
   prompt and retry generating the images with a new prompt that 
   incorporates the feedback.
   d. Maximum 1 retry.
4. After an image passes, list the image file path. A total of two images 
should pass.

IMPORTANT: brand_guide parameter in evaluate_image must always be the 
literal filename "guide.png", never the brand analysis text content.
"""

from google.adk.models import Gemini
class DLAIGemini(Gemini):
    _client: genai.Client = None  # class-level cache
    
    @property
    def api_client(self) -> genai.Client:
        if self._client is None:
            self._client = genai.Client(
                project=project_id,
                location="global",
                credentials=credentials,
                http_options=types.HttpOptions(
                    base_url=os.getenv("GOOGLE_VERTEX_BASE_URL")
                )
            )
        return self._client

agent = Agent(
    name="image_agent",
    model=DLAIGemini(model=TEXT_MODEL_ID),
    tools=[
        brand_analysis,
        generate_design_concepts,
        generate_idea_image,
        evaluate_image,
    ],
    instruction=SYSTEM_PROMPT,
)

### Run the agent

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Please wait:</b> The following notebook cell may take a few minutes to run.</p>

In [ ]:
runner = InMemoryRunner(agent=agent, app_name="image_agent")
session = await runner.session_service.create_session(
    app_name="image_agent", user_id="user"
)

prompt = """Create a UI landing page for this cooking app. 
The initial style guide image is in a file named guide.png"""

user_message = types.Content(
    role="user",
    parts=[types.Part(text=prompt)]
)
    
async for event in runner.run_async(
    user_id="user",
    session_id=session.id,
    new_message=user_message,
):
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text)
            if part.function_call:
                print(f"\u2192 Calling: {part.function_call.name}")

**Resources**
- Agent Development Kit (ADK) [documentation](https://adk.dev/)